# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tannusaini2110-spec/Internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [1]:

# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
%pip -q install duckdb huggingface_hub scikit-learn

import duckdb, os
import numpy as np
import pandas as pd
from google.colab import userdata
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLE = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

feat = con.sql(f"""
    SELECT content_hash_id, client_hash_id,
           AVG(gsc_avg_position) as avg_position,
           SUM(gsc_impressions) as total_impressions,
           SUM(gsc_clicks) as total_clicks,
           AVG(ga4_engaged_sessions) as avg_engaged_sessions,
           SUM(ga4_pageviews) as total_pageviews,
           COUNT(*) as days_seen
    FROM {TABLE}
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
""").df().fillna(0)

feat["ctr"] = np.where(feat["total_impressions"] > 0,
                        feat["total_clicks"] / feat["total_impressions"], 0)
feat["is_declining"] = (feat["total_clicks"] <= feat["total_clicks"].quantile(0.30)).astype(int)

feature_cols = ["avg_position", "total_impressions", "avg_engaged_sessions", "total_pageviews", "days_seen"]
X, y = feat[feature_cols], feat["is_declining"]

# Honest, client-grouped split -- same design validated in ML-09 (not naive random)
splitter = GroupShuffleSplit(test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(feat, groups=feat["client_hash_id"]))
X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1)
model.fit(X_tr, y_tr)

queue = feat.iloc[test_idx].copy()
queue["decline_probability"] = model.predict_proba(X_te)[:, 1]

# avg_position == 0 means "no data" (flyrank-data gotcha) -- exclude from the median
median_position = feat.loc[feat["avg_position"] > 0, "avg_position"].median()
median_impressions = feat["total_impressions"].median()
median_days_seen = feat["days_seen"].median()

def reason_code(row):
    good_pos = 0 < row["avg_position"] <= median_position
    zero_ctr = row["ctr"] == 0 and row["total_impressions"] > 0
    thin = row["total_impressions"] < median_impressions and row["days_seen"] < median_days_seen
    if good_pos and zero_ctr:
        return "CTR_OPPORTUNITY"
    elif thin:
        return "LOW_VISIBILITY_ACTIVITY"
    return "MODEL_FLAGGED_OTHER"

queue["reason_code"] = queue.apply(reason_code, axis=1)

ranked_queue = (
    queue[queue["decline_probability"] >= 0.5]
    .sort_values("decline_probability", ascending=False)
    .reset_index(drop=True)
)
ranked_queue.insert(0, "rank", ranked_queue.index + 1)

print(f"Ranked queue: {len(ranked_queue):,} of {len(queue):,} held-out pages flagged for review")
print(ranked_queue["reason_code"].value_counts())
print()
print(ranked_queue[["rank", "reason_code", "decline_probability", "avg_position", "total_impressions"]].head(10).to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Ranked queue: 25,210 of 43,264 held-out pages flagged for review
reason_code
LOW_VISIBILITY_ACTIVITY    9336
MODEL_FLAGGED_OTHER        8982
CTR_OPPORTUNITY            6892
Name: count, dtype: int64

 rank             reason_code  decline_probability  avg_position  total_impressions
    1 LOW_VISIBILITY_ACTIVITY             0.990071          78.0                1.0
    2 LOW_VISIBILITY_ACTIVITY             0.990071         262.0                1.0
    3 LOW_VISIBILITY_ACTIVITY             0.990071          83.0                1.0
    4 LOW_VISIBILITY_ACTIVITY             0.990071         153.0                1.0
    5 LOW_VISIBILITY_ACTIVITY             0.990071          88.0                1.0
    6 LOW_VISIBILITY_ACTIVITY             0.990071          96.0                1.0
    7 LOW_VISIBILITY_ACTIVITY             0.990071          99.0                1.0
    8 LOW_VISIBILITY_ACTIVITY             0.990071         167.0                1.0
    9 LOW_VISIBILITY_ACTIVITY             0.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
base_rate = y.mean()
honest_accuracy = model.score(X_te, y_te)

print(f"Base rate (share of pages labeled declining): {base_rate:.1%}")
print(f"Honest (client-grouped) accuracy: {honest_accuracy:.1%}")
print(f"Skill over base rate: {honest_accuracy - max(base_rate, 1 - base_rate):.1%} points")
print()
print(f"Queue flags {len(ranked_queue):,} of {len(queue):,} held-out pages "
      f"({len(ranked_queue) / len(queue):.1%}) for review -- not for automatic action.")

Base rate (share of pages labeled declining): 61.1%
Honest (client-grouped) accuracy: 85.3%
Skill over base rate: 24.2% points

Queue flags 25,210 of 43,264 held-out pages (58.3%) for review -- not for automatic action.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
borderline = queue[(queue["decline_probability"] >= 0.45) & (queue["decline_probability"] <= 0.55)]
low_trust = ranked_queue[ranked_queue["reason_code"] == "MODEL_FLAGGED_OTHER"]

print(f"Borderline pages (score 0.45-0.55, needs second look): {len(borderline):,}")
print(f"Low-trust queue entries (MODEL_FLAGGED_OTHER, no explainable driver): {len(low_trust):,} "
      f"of {len(ranked_queue):,} ({len(low_trust) / max(len(ranked_queue),1):.1%})")

Borderline pages (score 0.45-0.55, needs second look): 2,151
Low-trust queue entries (MODEL_FLAGGED_OTHER, no explainable driver): 8,982 of 25,210 (35.6%)


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.inspection import permutation_importance

perm = permutation_importance(model, X_te, y_te, n_repeats=10, random_state=42, n_jobs=-1)
importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": perm.importances_mean
}).sort_values("importance", ascending=False).reset_index(drop=True)

retrain_triggers = {
    "base_rate_drift_pct_points": 5,
    "honest_accuracy_drop_pct_points": 5,
    "dominant_feature": importance_df.iloc[0]["feature"],
    "scheduled_retrain": "monthly",
}

print("Feature importance this month (monitor the top feature closest for drift):")
print(importance_df.to_string(index=False))
print()
print("Retrain trigger config:", retrain_triggers)

Feature importance this month (monitor the top feature closest for drift):
             feature  importance
   total_impressions    0.235334
        avg_position    0.015410
     total_pageviews    0.014509
avg_engaged_sessions    0.005395
           days_seen    0.000023

Retrain trigger config: {'base_rate_drift_pct_points': 5, 'honest_accuracy_drop_pct_points': 5, 'dominant_feature': 'total_impressions', 'scheduled_retrain': 'monthly'}


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import json
import matplotlib.pyplot as plt

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

ranked_queue.to_csv("work/outputs/ranked_action_queue.csv", index=False)

metrics = {
    "label_definition": "is_declining = bottom 30% of total_clicks, March 2026 snapshot",
    "base_rate": round(float(base_rate), 4),
    "honest_client_grouped_accuracy": round(float(honest_accuracy), 4),
    "feature_importance": importance_df.set_index("feature")["importance"].round(4).to_dict(),
    "queue_size": int(len(ranked_queue)),
    "held_out_pages": int(len(queue)),
    "reason_code_counts": ranked_queue["reason_code"].value_counts().to_dict(),
    "borderline_0.45_0.55_count": int(len(borderline)),
}
with open("work/outputs/model_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

fig, ax = plt.subplots(figsize=(6, 4))
ax.barh(importance_df["feature"], importance_df["importance"])
ax.set_xlabel("Permutation importance")
ax.set_title("What drives the decline-risk model")
plt.tight_layout()
plt.savefig("work/figures/feature_importance.png", dpi=150)
plt.close()

print("Exported:")
print("  work/outputs/ranked_action_queue.csv  (gitignored, regenerated)")
print("  work/outputs/model_metrics.json       (commit this)")
print("  work/figures/feature_importance.png   (commit this)")

Exported:
  work/outputs/ranked_action_queue.csv  (gitignored, regenerated)
  work/outputs/model_metrics.json       (commit this)
  work/figures/feature_importance.png   (commit this)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.